<h1 style="color:#1f4e79">RACOI 원본 적재 및 품질검사</h1>
<p><b>이슈 5 · 첫 번째 PR</b></p>

## 이번 노트북에서 확인할 질문

1. 원본 파일을 바꾸지 않고 MariaDB raw 테이블에 모두 적재했는가?
2. 각 원본의 기간·행 수·프로그램 수는 무엇인가?
3. 전처리 전에 사용자 확인이 필요한 데이터 차이는 무엇인가?

원본 API의 단위가 달라 인터넷 반응은 주간, 통합 반응은 월간, 댓글은 프로그램·주차·유형별로 보관한다. 이 단계에서는 숫자와 이름을 고치지 않는다.

In [1]:
# 목적: 로컬 MariaDB 연결을 열어 raw 적재를 수행한다.
# 입력: .env의 MariaDB 접속 정보와 데이터/원본의 수집 메타데이터.
# 출력 단위: 데이터베이스 연결 1개. 기간: 원본 보유 전체 기간. NULL·0: 원본값을 변경하지 않는다.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == '노트북' else Path.cwd()))
from 수집.원본_적재 import connect, load_raw, table_counts

connection = connect()
cursor = connection.cursor()
cursor.execute('SELECT VERSION()')
print(f'MariaDB 연결 성공: {cursor.fetchone()[0]}')

MariaDB 연결 성공: 11.8.3-MariaDB


In [2]:
# 목적: 원본 파일과 수집 이력을 idempotent 방식으로 raw 테이블에 적재한다.
# 입력: 수집 메타데이터 JSONL, 주간·월간 XLSX, 댓글 JSON.
# 출력 단위: 원본 행 1건 또는 댓글 배열 원소 1건. 기간: 주간 202547~202647, 월간 202511~202610. NULL·0: 원문 그대로 보존한다.
result = load_raw(connection)
print(result)
print(table_counts(connection))

C:\Users\yooha\OneDrive\Desktop\git-hub-projects\Racoi\BrandCast\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


{'ingestion_events': 166, 'weekly_files': 56, 'monthly_files': 15, 'comment_files': 19, 'raw_internet_response': {'scanned': 8484, 'inserted': 0}, 'raw_total_response': {'scanned': 2797, 'inserted': 0}, 'raw_comments': {'keyword': 0, 'sentiment': 0, 'network': 0}}
{'raw_ingestion_log': 166, 'raw_internet_response': 8484, 'raw_total_response': 2797, 'raw_comment_keyword': 840, 'raw_comment_sentiment': 15, 'raw_comment_network': 59}


In [3]:
# 목적: 적재 후 기간과 기본 규모를 검증한다.
# 입력: raw_internet_response, raw_total_response, raw_comment_sentiment.
# 출력 단위: 원본 응답 테이블별 집계 1행. 기간: 각 원본의 전체 보유 기간. NULL·0: 결측과 0을 합치지 않는다.
queries = {
    '인터넷 반응(주간)': 'SELECT MIN(period_code), MAX(period_code), COUNT(DISTINCT period_code), COUNT(*), COUNT(DISTINCT program_name) FROM raw_internet_response',
    '통합 반응(월간)': 'SELECT MIN(period_code), MAX(period_code), COUNT(DISTINCT period_code), COUNT(*), COUNT(DISTINCT program_name) FROM raw_total_response',
    '댓글 감성 응답 주차 수': 'SELECT COUNT(DISTINCT response_nweek) FROM raw_comment_sentiment',
}
for label, sql in queries.items():
    cursor.execute(sql)
    print(f'{label}: {cursor.fetchone()}')

인터넷 반응(주간): ('202547', '202647', 53, 8484, 484)
통합 반응(월간): ('202511', '202610', 12, 2797, 483)
댓글 감성 응답 주차 수: (14,)


## 전처리 전 확인 결과

- 같은 `기간·프로그램·채널` 조합이 주간 154개, 월간 187개 있다. 같은 방송의 재방송·편성 차이인지, 다른 원본 응답 버전인지 확인하고 대표 행을 정해야 한다.
- 주간 53개와 월간 12개는 원본 API 단위가 다르므로 직접 조인하지 않는다.
- 댓글은 프로그램 ID 11487, 요청 주차 202646의 표본이며 감성 추이는 응답 주차 202633~202646을 포함한다.
- 수치 열의 빈값은 이 적재본에서 발견되지 않았지만, 숫자 변환·0의 의미·프로그램명 통합 규칙은 아직 적용하지 않았다.

다음 PR에서는 각 규칙을 적용하기 전에 `전처리 전 값 → 제안 규칙 → 전처리 후 값 → 영향 행 수` 순서로 나누어 보이고, **“이 기준으로 전처리할까요?”**라고 확인받는다.